In [1]:
print("hello")

hello


In [6]:
import os 
import csv 
import json
import uuid 
import re
import tiktoken
from tqdm import tqdm

In [3]:
tokenizer=tiktoken.get_encoding("cl100k_base")
def get_token_len(text:str)->int:
    """Helper to return the token length of a given text string."""
    return len(tokenizer.encode(text))
print("Prerequisites imported successfully!")

Prerequisites imported successfully!


In [4]:
def split_recursive_sentence(text:str,max_tokens:int=400,sentence_overlap:int=2)->list:
    """
    
    Splits text int o chunks of max_tokens,alighing cuts on sentence boundaries.

    """

    text=" ".join(text.split())
    if not text:
        return []

    raw_sentences=text.split(". ")
    sentences=[]
    for s in raw_sentences:
        s=s.strip()
        if s:
            if not s.endswith("."):
                s+="."
            sentences.append(s)
    chunks=[]
    i=0
    num_sentences=len(sentences)
    while i < num_sentences:
        current_chunk_sentencs=[]
        current_tokens=-0
        j=i
        while j<num_sentences:
            s_text=sentences[j]
            s_len=get_token_len(s_text)
            if s_len>max_tokens:
                if current_chunk_sentencs:
                    break
                words=s_text.split(" ")
                w_idx=0
                while w_idx<len(words):
                    sub_words=words[w_idx:w_idx+max_tokens]
                    chunks.append(" ".join(sub_words))
                    w_idx==max_tokens
                j+=1
                i=j
                break
            if current_tokens+s_len<=max_tokens:
                current_chunk_sentencs.append(s_text)
                current_tokens+=s_len
                j+=1
            else:
                break
        if current_chunk_sentencs:
            chunks.append(" ".join(current_chunk_sentencs))
        if j>=num_sentences:
            break
        i=max(j-sentence_overlap,i+1)
    return chunks

In [10]:
def stream_local_cuad(dataset_dir: str, limit: int = 100):
    # Support passing either the parent directory or the 'contracts' directory itself
    if os.path.basename(dataset_dir.rstrip("/")) == "contracts":
        contracts_dir = dataset_dir
    else:
        contracts_dir = os.path.join(dataset_dir, "contracts")
        
    if not os.path.exists(contracts_dir):
        raise FileNotFoundError(f"Contracts directory not found at: {contracts_dir}")
        
    count = 0
    for root, dirs, files in os.walk(contracts_dir):
        for file in sorted(files):
            if file.endswith(".txt"):
                if count >= limit:
                    break
                
                file_path = os.path.join(root, file)
                # In CUAD directory structure, the parent folder name represents the contract year
                year = os.path.basename(root)
                
                try:
                    with open(file_path, mode="r", encoding="utf-8") as f:
                        text = f.read()
                except Exception:
                    continue
                
                # Determine contract name/title from first non-empty lines
                lines = [line.strip() for line in text.split("\n") if line.strip()]
                contract_name = lines[0] if lines else "Unknown Contract"
                if len(contract_name) > 100:
                    contract_name = contract_name[:97] + "..."
                
                yield {
                    "doc_id": str(uuid.uuid5(uuid.NAMESPACE_DNS, file_path)),
                    "contract_name": contract_name,
                    "file_name": file,
                    "year": year,
                    "text": text
                }
                count += 1
        if count >= limit:
            break

In [17]:
def parse_cuad_document(doc: dict) -> list:                                                                                                                                    
        text = doc["text"]                                                                                                                                                         
        metadata = {                                                                                                                                                               
            "doc_id": doc["doc_id"],                                                                                                                                               
            "contract_name": doc["contract_name"],                                                                                                                                 
            "file_name": doc["file_name"],                                                                                                                                         
            "year": doc["year"]                                                                                                                                                    
        }                                                                                                                                                                          
        section_pattern = re.compile(r'^(SECTION\s+\d+\.\d+|ARTICLE\s+[IVXLCDM]+|EXHIBIT\s+[A-Z]|\bINDEMNITY\b|\bTERMINATION\b|\bLIMITATION\b)', re.IGNORECASE)                    
        lines = text.split("\n")                                                                                                                                                   
        current_section = "Preamble"                                                                                                                                               
        section_text_blocks = []                                                                                                                                                   
        current_block = []                                                                                                                                                         
                                                                                                                                                                                   
        for line in lines:                                                                                                                                                         
            stripped = line.strip()                                                                                                                                                
            if not stripped:                                                                                                                                                       
                continue                                                                                                                                                           
            # Check if line indicates a section boundary / heading                                                                                                                 
            if section_pattern.match(stripped) and len(stripped) < 80:                                                                                                             
                if current_block:                                                                                                                                                  
                    section_text_blocks.append((current_section, "\n".join(current_block)))                                                                                        
                    current_block = []                                                                                                                                             
                current_section = stripped
            else:
                current_block.append(stripped)
                
        if current_block:
            section_text_blocks.append((current_section, "\n".join(current_block)))  # <-- Fixed here
            
        all_chunks = []
        for section_name, section_text in section_text_blocks:
            if len(section_text.strip()) > 50:
                chunks = split_recursive_sentence(section_text)
                for chunk_text in chunks:
                    chunk = {
                        "chunk_id": str(uuid.uuid4()),
                        "chunk_text": chunk_text,
                        "section": section_name
                    }
                    chunk.update(metadata)
                    all_chunks.append(chunk)
    
        return all_chunks

In [18]:
DATASET_PATH = "/Users/mast/Documents/VInayPrograming/RAG/dataset"

print(f"Reading local CUAD documents from: {DATASET_PATH}")
sample_chunks = []

doc_stream = stream_local_cuad(DATASET_PATH, limit=3)
for doc in tqdm(doc_stream, total=3, desc="Parsing contract text"):
    doc_chunks = parse_cuad_document(doc)
    sample_chunks.extend(doc_chunks)

print(f"\nGenerated a total of {len(sample_chunks)} Sentence-Aligned chunks from contracts.")

Reading local CUAD documents from: /Users/mast/Documents/VInayPrograming/RAG/dataset


Parsing contract text: 100%|██████████| 3/3 [00:00<00:00, 32.59it/s]


Generated a total of 66 Sentence-Aligned chunks from contracts.


In [19]:
if sample_chunks:
    sample=sample_chunks[0]
    print("=== INSPECTING LOCAL SAMPLE CHUNK ===")
    print(f"Contract Name : {sample['contract_name']}")
    print(f"File Name     : {sample['file_name']}")
    print(f"Year          : {sample['year']}")
    print(f"Section Name  : {sample['section']}")
    print(f"Chunk ID      : {sample['chunk_id']}")
    print(f"Chunk Tokens  : {get_token_len(sample['chunk_text'])} tokens")
    print("\n--- Chunk Text ---")
    print(sample['chunk_text'])
else:
    print("No chunks generated. Make sure CUAD is downloaded to the dataset/ contracts directory.")

=== INSPECTING LOCAL SAMPLE CHUNK ===
Contract Name : Exhibit 10.19
File Name     : 000000000.txt
Year          : 2013
Section Name  : Preamble
Chunk ID      : 6e731803-3247-44ef-8009-5e71dc688d18
Chunk Tokens  : 359 tokens

--- Chunk Text ---
Exhibit 10.19 Agreement STEINWAY & SONS WITH LOCAL 81102, F.W. I.U.E.-C.W.A, A.F.L., C.I.O. JANUARY 1, 2013 -------------------------------------------------------------------------------- AGREEMENT made as of January 1, 2013 between Steinway, Inc. d.b.a. Steinway & Sons, (the “Company”) and Local 81102, F.W.,I.U.E.-C.W.A. AFL-CIO, (the “Union”.) WITNESSETH: WHEREAS, it is the intent and desire of both the Company and the Union to cooperate with each other in the administration of the provisions of this Agreement in order to achieve more stable and desirable conditions of employment for the employees covered hereby and more harmonious and profitable operations for the Company; NOW, THEREFORE, in consideration of these premises, the parties hereto